In [1]:
import pandas as pd
import numpy as np

#load dataset
df=pd.read_csv('buddymove_holidayiq.csv')
df.head()

,User Id,Sports,Religious,Nature,Theatre,Shopping,Picnic
0,User 1,2,77,79,69,68,95
1,User 2,2,62,76,76,69,68
2,User 3,2,50,97,87,50,75
3,User 4,2,68,77,95,76,61
4,User 5,2,98,54,59,95,86


In [2]:
#print the names of the columns
print(df.columns.tolist())

['User Id', 'Sports', 'Religious', 'Nature', 'Theatre', 'Shopping', 'Picnic']


In [3]:
#this dataset has no label, so we make one:
#label 1 = high Nature rating (>= median), 0 = low Nature rating
label=(df['Nature']>=df['Nature'].median()).astype(int)
print(label.value_counts())

Nature
1    126
0    123
Name: count, dtype: int64


In [4]:
#print the top five records
print(df.head())

  User Id  Sports  Religious  Nature  Theatre  Shopping  Picnic
0  User 1       2         77      79       69        68      95
1  User 2       2         62      76       76        69      68
2  User 3       2         50      97       87        50      75
3  User 4       2         68      77       95        76      61
4  User 5       2         98      54       59        95      86


In [5]:
#features = the other 5 ratings (Nature is removed to avoid leakage)
X=df[['Sports','Religious','Theatre','Shopping','Picnic']].values
y=label.values

In [6]:
#print data(feature) shape
print(X.shape)

(249, 5)


In [7]:
#print target(or label) shape
print(y.shape)

(249,)


In [8]:
#Import train_test_split function
from sklearn.model_selection import train_test_split

#70% training and 30% test
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [9]:
#scale features, then K=5
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn=make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train,y_train)
y_pred=knn.predict(X_test)

In [10]:
from sklearn import metrics
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))

Accuracy: 0.8666666666666667


In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"Precision: {precision_score(y_test, y_pred, average='weighted')}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted')}")
print(f"F1-score: {f1_score(y_test, y_pred, average='weighted')}")

Precision: 0.8765503875968993
Recall: 0.8666666666666667
F1-score: 0.8659523809523809


In [12]:
print(metrics.confusion_matrix(y_test, y_pred))
print(metrics.classification_report(y_test, y_pred, target_names=['Low Nature','High Nature']))

[[35  2]
 [ 8 30]]
              precision    recall  f1-score   support

  Low Nature       0.81      0.95      0.88        37
 High Nature       0.94      0.79      0.86        38

    accuracy                           0.87        75
   macro avg       0.88      0.87      0.87        75
weighted avg       0.88      0.87      0.87        75



Conclusion:


The KNN model (K=5) used the ratings of the other 5 categories to predict whether a user's Nature rating is at or above the median (High) or below it (Low), and it reached about 86.7% accuracy. The two classes are almost equal in size, so guessing would give about 50%, which makes this a fairly good result. The model found almost all Low Nature users (recall 0.95), but it missed about 1 in 5 High Nature users (recall 0.79). The label was made by us using the median, so users near the cutoff are hard to classify, and the test set has only 75 users, so the score can change with a different split. Cross-validation and tuning K would give a more stable result